# Ridge, Lasso, and Elastic Net 📉

## How to stop a model from trying *too* hard



Remember our very first regression example: predicting **Vickers hardness** from **carbon content**. We drew a straight line through the points. To choose that line, we picked the coefficients that made the **sum of squared residuals (SSR)** as small as possible.

$$
\text{SSR} = \sum_i \left( y_i - \hat{y}_i \right)^2
\quad\longrightarrow\quad \text{minimize}
$$

This method is called **Ordinary Least Squares (OLS)**. It is tight and simple: it just tries to sit as close to every training point as it can.


:::{note}important
"best fit on the data I already have" is **not** the same as "best guess for data I have not seen yet."
:::

When you train a model on the data that you have, and if you fit the regression line 'perfectly' on to that data - what are the chances that when you use your model on 'unknown' data for predictions- would it work?

Remember, that getting it right on 'test' data is important in materials, because if you want to discover new materials that are still unknown - that's test data!

Let's see how we can approach this.



## ❗Problem 1: A perfect fit can be a trap

Imagine two students preparing for an exam 	📖

- **Student A** memorizes every past-paper answer, word for word.
- **Student B** learns the *ideas* behind the questions.

On the past papers, Student A looks perfect. But on the *real* exam, with new questions, Student B usually wins. Student A memorized the noise. Student B learned the pattern.

Ordinary Least Squares (OLS) can behave like Student A. If we give it enough features, it will bend and stretch to touch the training points as closely as possible - including the random noise in that data. This is called **overfitting**.

An overfitted model often has one important sign: **very large coefficients**. To fit to every point, the model turns some features up loud, then turns others down to cancel them out. The numbers get big and jumpy.

This is just like doing a group project with a very talkative and loud team-mate 📣 : he might drive the team decisions just because he can get up and talk better than others, whereas the team members with real smart ideas does not get a chance.

:::{note} The big idea
If large coefficients are a sign of trying too hard, then let's gently **discourage large coefficients**. We do this by adding a small **penalty** for size. That gives rise to Ridge, Lasso, and Elastic Net, where Elastic Net is actually a combination of the best parts of both Ridge and Lasso.
:::



## ❗Problem 2: Features that say the same thing (multicollinearity) 👫

There is a second reason coefficients blow up. It happens when features are **correlated** - when they carry almost the same information.

A everyday example: house price 🏠.

> The price of a house depends on floor area, number of bedrooms, number of bathrooms, garage size, location, and so on.
> But bigger houses tend to have *more* bedrooms **and** more bathrooms **and** more floor area. These features move together.

If you ask OLS, "how much is floor area worth, and how much is bedrooms worth, *separately*?", it struggles. When two features rise and fall together, the model cannot cleanly split the credit between them. It may hand one a huge positive coefficient and the other a huge negative one that almost cancels out. The fit looks fine, but the coefficients are unstable - if you change the data a little, they will swing wildly.

This overlap-of-information is called **multicollinearity**.

:::{admonition} Why materials scientists really care about this
:class: seealso
In materials data, correlated descriptors are *everywhere*:

- Atomic mass, atomic radius, atomic number
- Electronegativity vs. ionization energy
- Density vs. atomic mass and atomic volume
- Thermal conductivity vs. electrical conductivity

We often use dozens of such descriptors in a model. Many of them are telling us the same thing. Plain OLS does not work well in that situation. Our new methods will.
:::





## 🔧Fix 1: Ridge Regression - keep the coefficients on a leash ➰

Ridge keeps the same goal as OLS (fit the data), but adds one extra rule: **keep the coefficients small🔻**.

We do this by adding a penalty equal to the **sum of the squared coefficients**, multiplied by a strength knob we call **λ** (lambda):

$$
\underbrace{\text{SSR}}_{\text{fit the data}} \;+\; \underbrace{\lambda \sum_{j} w_j^{2}}_{\text{penalty for big coefficients}} \quad\longrightarrow\quad \text{minimize}
$$

Because the penalty uses **squared** coefficients, it is called the **L2 penalty**.

Think of λ as a tax collector. The model *wants* large coefficients to fit the noise. The penalty makes large coefficients **expensive**. So the model settles for smaller, calmer coefficients - unless a feature is truly worth the cost.

:::{admonition} What the λ knob does
:class: tip
- **λ = 0** → no penalty at all → we are back to ordinary OLS.
- **small λ** → gentle nudge → coefficients shrink a little.
- **large λ** → strong pull → coefficients shrink a lot, all toward zero.

Ridge shrinks coefficients toward zero, but it **never sets them exactly to zero**. Every feature stays in the model, just quieter.
:::

You can try changing the λ in the [interactive simulation](data/ridge_explorer.html) to see the effect.

Why is smaller better? We are making a trade ⚖️. We accept a slightly *worse* fit on the training data (a little more **bias**) in exchange for coefficients that are stable and don't overreact to noise (a lot less **variance**). The reward is a model that **generalizes** better - it does a better job on new, unseen materials.

This is also Ridge's superpower against **multicollinearity**. Instead of giving one correlated feature a huge coefficient and its twin a huge negative one, Ridge **shares the weight** between them and keeps both small. Stable and sensible.

:::{important} Two housekeeping rules (do these before every Ridge/Lasso/Elastic Net)
1. **Scale your features first.** 📏The penalty punishes big coefficients - but a feature *(e.g. bandgap)* measured in large units $(eV)$ naturally gets a small coefficient, while a feature in tiny units $(J)$ gets a large one. Without scaling, the penalty is unfair. Standardize your features (the same lesson from the clustering chapter!).
2. **We usually do not penalize the intercept** 📈 $w_0$. The intercept just sets the baseline height of the line; shrinking it would not help.
:::



## 🔧Fix 2: Lasso Regression - shrink some coefficients all the way to zero 0️⃣

Ridge makes coefficients small. But,

> What if we shrink some coefficients so hard that they become **exactly zero**? 0️⃣

A coefficient of exactly zero means the feature is **switched off**. It no longer affects the prediction at all. In effect, the model has *dropped* that feature. That is **feature selection**, and it can be done automatically.

To get this behavior, Lasso changes the penalty. Instead of squared coefficients, it uses the **absolute value** of the coefficients:

$$
\text{SSR} \;+\; \lambda \sum_{j} \left| w_j \right| \quad\longrightarrow\quad \text{minimize}
$$

This is the **L1 penalty**. And that small change - squares becoming absolute values - has a surprisingly large effect.

:::{dropdown} Why does L1 give *exact* zeros, but L2 does not? 
Picture the penalty as a boundary the coefficients must live inside.

- The **L2 (Ridge)** boundary is a smooth circle. The best solution usually lands somewhere on the smooth edge - rarely exactly on an axis. So coefficients get small, but not zero.
- The **L1 (Lasso)** boundary is a diamond with **sharp corners**, and those corners sit *right on the axes*. The best solution often lands on a corner - and a corner means one of the coefficients is exactly zero.

You do not need the geometry to use Lasso. Just remember: **L1 has corners, and corners create zeros.**
:::

So when is Lasso the right tool? **When you suspect that many of your features are simply irrelevant.** Lasso will quietly find them and set their coefficients to zero, leaving you with a shorter, cleaner list of features that actually matter. For a materials dataset with dozens of descriptors, that is a very useful thing.




## Ridge vs. Lasso: pick your strength

Both methods add a penalty. But they behave differently, and each has a weak spot.

| | **Ridge (L2)** | **Lasso (L1)** |
|---|---|---|
| Penalty | sum of **squared** coefficients | sum of **absolute** coefficients |
| Shrinks coefficients? | Yes, toward zero | Yes, some *to* zero |
| Drops features? | No - keeps all of them | Yes - automatic feature selection |
| Correlated features? | Handles them well (shares the weight) | Struggles - tends to pick one and drop its twins, a bit at random |
| Best when… | many features matter, and they overlap | many features are irrelevant |

👀 Look at that last row carefully, that helps you decide which method you should use.

- **Ridge** is great with **correlated (multicollinear)** features, but it can't do feature selection.
- **Lasso** is great at **feature selection**, but it is shaky when features are correlated - faced with a group of near-twin features, it often keeps one and zeros the rest, more or less by luck.

So each tool fixes one problem and stumbles on the other.

:::{admonition} The obvious wish
:class: important
Can we have **both**? Feature selection *and* graceful handling of correlated features?
:::

Yes. That is exactly what Elastic Net is for.🥳 



## 🔧Fix 3: Elastic Net - the best of both 🤗

Elastic Net simply uses **both penalties at once** - the L1 (Lasso) part and the L2 (Ridge) part:

$$
\text{SSR} \;+\; \lambda_1 \sum_j |w_j| \;+\; \lambda_2 \sum_j w_j^{2}
$$

The L1 part gives you feature selection. The L2 part gives you the stability with correlated features. Together, when several features are near-twins, Elastic Net tends to keep or drop the **whole group together**, instead of picking one at random the way Lasso does.

There is a small cost: now we have **two** strengths to choose, $\lambda_1$ and $\lambda_2$. To make life easier, we usually rewrite it with one overall strength $\lambda$ and one **mixing knob** $\alpha$:

$$
\text{SSR} \;+\; \lambda \Big[\, \alpha \sum_j |w_j| \;+\; (1-\alpha)\sum_j w_j^{2} \,\Big]
$$

where

- $\lambda$ = the **overall regularization strength** (how hard we penalize, in total).
- $\alpha$ = the **mixing parameter**, between 0 and 1:
  - $\alpha = 1$ → **pure Lasso**
  - $\alpha = 0$ → **pure Ridge**
  - $0 < \alpha < 1$ → **a blend of both**

Elastic Net gives you the whole dial in between.🎛️

:::{warning} A naming trap when you reach the code
The **lecture** uses $\lambda$ for overall strength and $\alpha$ for the mix.
But **scikit-learn uses different names**, and they clash:

- scikit-learn's `alpha` = our $\lambda$ (overall strength).
- scikit-learn's `l1_ratio` = our $\alpha$ (the Lasso/Ridge mix).

So in code, `alpha` is the strength, and `l1_ratio` is the blend. Keep this card next to your keyboard.
:::

---

## How do we choose the knobs? Cross-validation

We now have knobs to set: λ (and, for Elastic Net, the mix too). How do we pick good values without cheating by peeking at the test set?

The answer is **cross-validation (CV)**. The idea is to try each candidate value and give it a *fair* exam on data it did not train on.

Here is the recipe (this is what `RidgeCV`, `LassoCV`, and `ElasticNetCV` do for you):

1. Make a list of candidate values, e.g. `[0.001, 0.01, 0.1, 1.0, 10.0]`.
2. For **each** candidate value:
   - Split the training data into $k$ folds (say $k = 5$).
   - Take turns: train on $k-1$ folds, then score on the one fold that was held out.
   - Average the scores across all $k$ folds. That average is the value's "report card."
3. Compare the averages across all candidates.
4. Keep the value with the **best average score** (lowest error / highest R²).

Because every point gets a turn as the held-out fold, no data is wasted, and no value gets to cheat.

:::{admonition} Illustrative example (made-up numbers, for the shape of it)
:class: note
The scores below are **invented to show the idea** - they are *not* results from our dataset.

| λ (alpha) | Fold 1 | Fold 2 | … | Fold k | **Average** |
|---|---|---|---|---|---|
| 0.001 | 0.82 | 0.85 | … | 0.84 | **0.835** |
| 0.01  | 0.80 | 0.83 | … | 0.82 | **0.825** |
| 0.1   | 0.78 | 0.80 | … | 0.79 | **0.790** |

Here the winner is **λ = 0.001**, because it has the highest average score. In practice, run it on the real data and read off the winner - don't assume these numbers.
:::

:::{admonition} In the notebook
:class: note
Use `RidgeCV` and `LassoCV` to let cross-validation choose λ for you. Then extend the same notebook and fit **`ElasticNetCV`**, which searches over *both* the strength and the mix. Compare the three models' coefficients and their scores on the held-out data.
:::

---

## Putting the story together

Let's zoom back out. Every step of this chapter came from one worry: **OLS can try too hard and fail on new data.**

1. We noticed that trying too hard shows up as **large, jumpy coefficients** - made worse by **correlated features**.
2. **Ridge** adds an **L2 penalty** to keep coefficients small and stable. Great for correlated features. Keeps every feature.
3. **Lasso** adds an **L1 penalty** that pushes some coefficients **exactly to zero**. Great for dropping useless features. Wobbly when features are correlated.
4. **Elastic Net** uses **both penalties**, with a mixing knob α. It selects features *and* stays stable with correlated ones - Ridge and Lasso are just its two extreme settings.
5. We choose the knobs honestly with **cross-validation** (`RidgeCV`, `LassoCV`, `ElasticNetCV`).

One sentence to carry with you: **we give up a tiny bit of fit on today's data to buy a lot of reliability on tomorrow's data.**

---

## Check your understanding

:::{admonition} **Question 1**. 
A linear regression is trained on 20 steel alloys using 15 standardised composition features to predict yield strength. In the training set, yield strength has a standard deviation of about 50 MPa. The model scores training R² = 0.99 but test R² = 0.21. Part of its coefficient table is shown below.

| Feature | Coefficient (MPa per SD of feature) |
|---|---:|
| Cr | +412.3 |
| Mo | −398.7 |
| Ni | +6.1 |
| C | +31.5 |

In this dataset, Cr and Mo have a correlation of 0.97.

Which entries in the table look suspicious, and why?
:::

:::{dropdown} Answer
The **Cr** and **Mo** coefficients are suspicious, for two reasons.

1. **They are implausibly large.** The features are standardised, so each coefficient tells us how much the prediction changes when that feature moves by one standard deviation. A change of about 400 MPa is roughly 8 standard deviations of the target. No single element should have that much effect.
2. **They have opposite signs on almost identical features.** Cr and Mo move together (correlation 0.97), so on the training alloys their large contributions mostly cancel. On new alloys, where the two don't move in exactly the same way, the cancellation breaks and predictions go wrong.

This is the pattern regularisation is designed to fix. Ridge adds a penalty on large coefficients, which shrinks both Cr and Mo toward similar, moderate values.

**Why standardising matters here:** without standardised features, the size of a coefficient depends on the units. If carbon were given as a mass fraction instead of wt%, its coefficient would be 100 times larger with no change in the model. Standardising the features, and comparing against the spread of the target, gives us a fair yardstick for calling a coefficient "large".
:::

:::{admonition} **Questions 2**. 
You have 40 materials descriptors and you strongly suspect most of them are useless. Which method is the natural first choice, and why?
:::
:::{dropdown} Answer
**Lasso.** Its L1 penalty can set coefficients **exactly to zero**, switching off the useless descriptors automatically (feature selection).
:::

:::{admonition} **Question 3**. 
Atomic mass, atomic radius, and atomic number are all in your feature list and move together. Plain Lasso keeps giving you unstable results - sometimes one is chosen, sometimes another. What should you reach for?
:::
:::{dropdown} Answer
**Elastic Net.** Its L2 part stabilizes correlated features and tends to keep or drop the whole correlated group together, instead of picking one at random the way pure Lasso does.
:::

:::{admonition} **Question 4**. 
In scikit-learn, you set `ElasticNet(alpha=..., l1_ratio=...)`. Which one is the overall strength, and which is the Ridge/Lasso mix?
:::
:::{dropdown} Answer
`alpha` is the **overall strength** (our λ). `l1_ratio` is the **mix** (our α): `l1_ratio=1` is pure Lasso, `l1_ratio=0` is pure Ridge.
:::

---

:::{seealso} Where these ideas come from (optional history)
For students who like to see the original sources:

- **Ridge regression** - Hoerl, A. E. & Kennard, R. W. (1970). *Ridge Regression: Biased Estimation for Nonorthogonal Problems.* Technometrics, 12(1), 55–67.
- **Lasso** - Tibshirani, R. (1996). *Regression Shrinkage and Selection via the Lasso.* Journal of the Royal Statistical Society, Series B, 58(1), 267–288.
- **Elastic Net** - Zou, H. & Hastie, T. (2005). *Regularization and Variable Selection via the Elastic Net.* Journal of the Royal Statistical Society, Series B, 67(2), 301–320.

These are the papers that introduced each method. You do not need them to do the coursework - they are here in case you want to go to the source.
:::

:::{admonition} Coming up
:class: tip
Later in the course we will revisit Ridge and Lasso from a **Bayesian** angle - where the penalty turns out to be a *prior belief* that coefficients should be small. Same idea, a new and rather elegant way of seeing it.
:::
